In [4]:
# loading the env variables
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
# initialize the qdrant client
from qdrant_client import QdrantClient
import os

client = QdrantClient(
    url=os.getenv('QDRANT_CLOUD_URL'),
    api_key=os.getenv('QDRANT_CLOUD_API_KEY'),
    prefer_grpc=False
) 

In [6]:
# initializing the embedding model

from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs={'device': 'cpu'}
encode_kwargs={'normalize_embeddings' : 'False'}
embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3033.96it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# initializing the vector store and retriever

from langchain_qdrant import QdrantVectorStore
from qdrant_client.models import PayloadSchemaType

vectorstore =QdrantVectorStore(
    embedding=embedding,
    client=client,
    collection_name=os.getenv('COLLECTION_NAME')
)

client.create_payload_index(
    collection_name=os.getenv("COLLECTION_NAME"),
    field_name="metadata.section",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=20, status=<UpdateStatus.COMPLETED: 'completed'>)

In [33]:
import re
from qdrant_client.models import Filter, FieldCondition, MatchValue
from sentence_transformers import CrossEncoder
from typing import List, Optional
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

reranker = CrossEncoder("BAAI/bge-reranker-base")

def extract_section(query: str):
    match = re.search(r'\b(?:section|sec|s\.)\s*(\d+)\b', query.lower())
    return match.group(1) if match else None

retriever = vectorstore.as_retriever(
        search_type = 'similarity',
        search_kwargs= {'k':10}
    )
    
def retrieve(query: str) -> List[Document]:
    
    section = extract_section(query)

    if section:
        docs = vectorstore.similarity_search(
            query,
            k=1,
            filter=Filter(
                must=[
                    FieldCondition(
                        key="metadata.section",
                        match=MatchValue(value=section)
                    )
                ]
            )
        )
    
        if docs:
            return docs 
    
    docs = retriever.invoke(query)
    pairs = [[query, d.page_content] for d in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )
    return [d for _, d in ranked[:3]]

retrieval_chain = RunnableLambda(retrieve)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6043.93it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
def format_docs(docs: List[Document]) -> str:
    """Format documents into context string"""
    if not docs:
        return "No relevant sections found."
    
    context_parts = []
    for doc in docs:
        section = doc.metadata.get('section', 'Unknown')
        content = doc.page_content.strip()
        context_parts.append(f"BNS Section {section}:\n{content}")
    
    return "\n\n" + "="*50 + "\n\n".join(context_parts)

In [43]:
result = format_docs(retrieval_chain.invoke('What is the section 376 of BNS'))
result

'\n\n==================================================BNS Section 64:\nBharatiya Nyaya Sanhita S. 64 Punishment for rape. Description IPC Section 376 Explanation: For the purposes of this sub-section, “armed forces” means the naval, army and air forces and includes any member of the Armed Forces constituted under any law for the time being in force, including the paramilitary forces and any auxiliary forces that are under the control of the Central Government or the State Government; “hospital” means the precincts of the hospital and includes the precincts of any institution for the reception and treatment of persons during convalescence or of persons requiring medical attention or rehabilitation; “police officer” shall have the same meaning as assigned to the expression “police” under the Police Act, 1861; “women’s or children’s institution” means an institution, whether called an orphanage or a for neglected women or children or a widow’s or an institution called by any other name, 

In [44]:
from dotenv import load_dotenv
load_dotenv()

True

In [45]:
from langchain_groq import ChatGroq
import os 

model = ChatGroq(
    model=os.getenv('GROQ_MODEL_NAME'),
    api_key=os.getenv('GROQ_API_KEY')
)

In [46]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
 
prompt = PromptTemplate(
    template="""
You are LegalQuery, an AI legal information assistant.

STRICT RULES
- Provide information only for educational purposes.
- Always begin with a legal disclaimer.
- Use ONLY the provided context.
- Do NOT assume jurisdiction.
- Do NOT generate legal sections, punishments, or procedures that are not present in the context.
- If the answer is not in the context, respond exactly:
"The requested information is not available in the provided legal context."

TASK
1. Analyze the user query.
2. Identify if the query asks about:
   - a legal section
   - punishment or penalty
   - definition of an offence
   - explanation of a law
3. Extract the relevant information from the context.
4. Provide a clear explanation strictly based on the context.

CONTEXT
{context}

USER QUERY
{query}

RESPONSE FORMAT

Disclaimer:
<short legal disclaimer>

Relevant Section:
<section number if present in context>

Quoted Law:
<exact excerpt from context>

Explanation:
<clear explanation derived only from the context>
""",
    input_variables=["context", "query"]
)

In [47]:
parser = StrOutputParser()

In [48]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {
        'context' : retrieval_chain | format_docs,
        'query' : RunnablePassthrough()
    } | prompt | model | parser
)

In [50]:
result = chain.invoke('A girl was harassed at the college by a male faculty.')
result_list = result.split('\n')
for res in result_list:
    print(res)

Disclaimer: The provided information is for educational purposes only and may not be applicable in all jurisdictions. The interpretation of laws may vary based on specific circumstances and jurisdiction.

Relevant Section: BNS Section 75

Quoted Law: A man committing any of the following act: physical contact and advances involving unwelcome and explicit sexual overtures; or a demand or request for sexual favours; or showing pornography against the will of a woman; or making sexually coloured remarks, shall be guilty of the offence of sexual harassment.

Explanation: According to BNS Section 75, a man who commits any of the acts mentioned such as physical contact and advances involving unwelcome and explicit sexual overtures, demand or request for sexual favours, showing pornography against the will of a woman, or making sexually coloured remarks shall be guilty of the offence of sexual harassment. In the given scenario, if a male faculty harassed a girl at the college, he may be found